# Shadow Tomography 教程（分层调用）

本教程展示同一件事的三种层次：
1. 高层：直接使用 `ShadowTomography.run`
2. 中层：手动选择 backend，然后调用 `run_shadow_with_backend`
3. 底层：从 `samples + basis_patterns` 手动调用 `estimate_observables`

默认使用 `Simulator`，便于本地快速验证。

In [3]:
from quantum_hw import QuantumHardwareClient, ShadowTomography
from quantum_hw.api.backend import Backend
from quantum_hw.algorithms.shadow import run_shadow_with_backend, estimate_observables

import numpy as np

def section(title: str):
    print("\n" + "=" * 18, title, "=" * 18)

In [4]:
section("shared setup")
client = QuantumHardwareClient()
num_qubits = 6
circuit = client.build_circuit("ghz", num_qubits=num_qubits, measure=False)
observables = ["ZZIIII", "IIZZII", "IIXXII"]

print("num_qubits:", num_qubits)
print("observables:", observables)


================== shared setup ==================
num_qubits: 6
observables: ['ZZIIII', 'IIZZII', 'IIXXII']


## 1) 高层入口：`ShadowTomography.run`

这是最推荐给业务用户的方式：自动做芯片筛选 + 执行 + 结果封装。

In [5]:
section("high-level api")
shadow = ShadowTomography(client=client, seed=42)

result_high = shadow.run(
    circuit=circuit,
    name="tutorial_shadow_high",
    num_qubits=num_qubits,
    shots=2048,
    shots_per_basis=4,
    observables=observables,
    estimator="mom",
    mom_groups=16,
    prefer_chips="Simulator",
)
print("task_ids:", result_high.task_ids)
print("num_samples:", result_high.num_samples)
print("observable_estimates (final):", result_high.observable_estimates)


================== high-level api ==================
[shadow] read hardware information and select provider=quafu
task_ids: []
num_samples: 2048
observable_estimates (final): {'ZZIIII': 1.16015625, 'IIZZII': 1.0546875, 'IIXXII': -0.52734375}


## 2) 中层入口：手动拼接 backend + `run_shadow_with_backend`

这层适合做“可控性更强”的优化，比如自己决定芯片、排序策略、重试策略。

In [6]:
section("manual backend + run_shadow_with_backend")

# 你可以传 prefer_chips=["Dongling", "Baihua"]，这里为了稳定演示固定 Simulator

chip_name = "Simulator"
backend = Backend(chip_name)

result_manual_backend = run_shadow_with_backend(
    client=client,
    qc=circuit,
    name="tutorial_shadow_manual_backend",
    num_qubits=num_qubits,
    backend=backend,
    chip_name=chip_name,
    shots=2048,
    shots_per_basis=4,
    observables=observables,
    estimator="mean",
    seed=123,
)

print("chosen chip:", chip_name)
print("num_samples:", result_manual_backend.num_samples)
print("observable_estimates:", result_manual_backend.observable_estimates)


================== manual backend + run_shadow_with_backend ==================
chosen chip: Simulator
num_samples: 2048
observable_estimates: {'ZZIIII': 0.984375, 'IIZZII': 0.87890625, 'IIXXII': -0.4921875}


## 3) 中层入口：调用 `estimate_observables`

把上一节返回的采样数据当作输入，自己控制估计器。

In [7]:
section("lowest-level estimator")
samples = np.asarray(result_manual_backend.samples, dtype=int)
basis_patterns = result_manual_backend.basis_patterns

est_mean, stderr_mean = estimate_observables(
    samples=samples,
    basis_patterns=basis_patterns,
    observables=observables,
    num_qubits=num_qubits,
    estimator="mean",
)

est_mom, stderr_mom = estimate_observables(
    samples=samples,
    basis_patterns=basis_patterns,
    observables=observables,
    num_qubits=num_qubits,
    estimator="mom",
    mom_groups=12,
)

print("mean estimator:", est_mean)
print("mean stderr:", stderr_mean)
print("mom estimator:", est_mom)
print("mom stderr:", stderr_mom)


================== lowest-level estimator ==================
mean estimator: {'ZZIIII': 0.984375, 'IIZZII': 0.87890625, 'IIXXII': -0.4921875}
mean stderr: {'ZZIIII': 0.062085482694384554, 'IIZZII': 0.05904996126674241, 'IIXXII': 0.06488171366699011}
mom estimator: {'ZZIIII': 0.9, 'IIZZII': 0.9473684210526315, 'IIXXII': -0.5541795665634675}
mom stderr: {'ZZIIII': 0.06591353603648042, 'IIZZII': 0.05903641989375138, 'IIXXII': 0.054133688285110386}


## 4) 演示真实硬件
在量子硬件上结合ZNE跑一遍（需要一点时间）

In [8]:
section("Real experiments")
shadow = ShadowTomography(client=client, seed=42)

result_high = shadow.run(
    circuit=circuit,
    name="tutorial_shadow_high",
    num_qubits=num_qubits,
    shots=2048,
    shots_per_basis=16, # 仅Baihua支持较小的shots_per_basis
    observables=observables,
    estimator="mom",
    mom_groups=16,
    prefer_chips="Baihua",
    zne=True,
)
print("num_samples:", result_high.num_samples)
print("observable_estimates (final):", result_high.observable_estimates)
print("observable_estimates_raw (raw):", result_high.observable_estimates_raw)


================== Real experiments ==================
[shadow] read hardware information and select provider=quafu
num_samples: 2048
observable_estimates (final): {'ZZIIII': 1.40625, 'IIZZII': 0.52734375, 'IIXXII': -0.193359375}
observable_estimates_raw (raw): {'ZZIIII': 1.265625, 'IIZZII': 0.421875, 'IIXXII': -0.10546875}


## 5) 手动构造随机测量线路 + `simulate_counts` 收集 `samples`

这一节手动随机抽取测量基，逐个构造“附加测量门”的线路，然后直接调用 simulator 底层采样函数。

In [11]:
section("manual random measurement + simulator sampling")
from quantum_hw.core.observables import append_measurement_basis
from quantum_hw.sim.statevector import simulate_counts
from quantum_hw.core.utils import get_samples

rng = np.random.default_rng(2026)
basis_choices = np.array(["X", "Y", "Z"], dtype=object)
num_basis = 2048
shots_per_basis = 16

manual_samples_blocks = []
manual_basis_patterns = []

for _ in range(num_basis):
    basis_pattern = rng.choice(basis_choices, size=num_qubits).tolist()
    qc_meas = circuit.deepcopy()
    append_measurement_basis(qc_meas, basis_pattern, target_qubits=list(range(num_qubits)))

    counts = simulate_counts(
        qc_meas,
        shots=shots_per_basis,
        seed=int(rng.integers(0, 2**32 - 1)),
    )
    block_samples = get_samples(counts, num_qubits=num_qubits)

    manual_samples_blocks.append(block_samples)
    manual_basis_patterns.extend([basis_pattern] * len(block_samples))

samples_manual = np.vstack(manual_samples_blocks).astype(int)

print("num_basis:", num_basis)
print("shots_per_basis:", shots_per_basis)
print("total manual samples:", samples_manual.shape[0])
print("sample shape:", samples_manual.shape)
print("first basis pattern:", manual_basis_patterns[0])
print("first sample row:", samples_manual[0].tolist())


================== manual random measurement + simulator sampling ==================
num_basis: 2048
shots_per_basis: 16
total manual samples: 32768
sample shape: (32768, 6)
first basis pattern: ['Z', 'X', 'X', 'Y', 'Y', 'Y']
first sample row: [0, 0, 0, 0, 0, 0]


## 6) 从 `samples` 手工计算 estimator（拆解版）

这一节把classical shadow (random Pauli measurements) 背后的核心原理展开写一遍，再和库函数结果做对比。

In [12]:
section("manual estimator from samples")
from quantum_hw.core.observables import pauli_basis_pattern

def estimate_from_samples_manual(
    samples: np.ndarray,
    basis_patterns,
    observables,
    num_qubits: int,
 ):
    nshots = samples.shape[0]
    estimates = {}
    stderrs = {}

    for obs in observables:
        obs_pattern = pauli_basis_pattern(obs, num_qubits=num_qubits)
        support = [i for i, op in enumerate(obs_pattern) if op != "I"]

        sample_values = np.zeros(nshots, dtype=float)
        for idx, (sample_row, basis_row) in enumerate(zip(samples, basis_patterns)):
            basis_match = True
            for q in support:
                if basis_row[q] != obs_pattern[q]:
                    basis_match = False
                    break
            if not basis_match: # 仅当样本的测量基底在支持集上与可观测量匹配时才计算估计值，否则该样本对该可观测量不提供信息，直接跳过
                continue

            value = 1.0
            for q in support:
                eigen = 1.0 - 2.0 * float(sample_row[q])
                value *= 3.0 * eigen
            sample_values[idx] = value

        estimates[obs] = float(sample_values.mean())
        stderrs[obs] = float(sample_values.std(ddof=1) / np.sqrt(nshots))

    return estimates, stderrs

manual_est, manual_stderr = estimate_from_samples_manual(
    samples=samples_manual,
    basis_patterns=manual_basis_patterns,
    observables=observables,
    num_qubits=num_qubits,
 )

lib_est, lib_stderr = estimate_observables(
    samples=samples_manual,
    basis_patterns=manual_basis_patterns,
    observables=observables,
    num_qubits=num_qubits,
    estimator="mean",
 )

print("manual estimates:", manual_est)
print("library estimates:", lib_est)
print("manual stderr:", manual_stderr)
print("library stderr:", lib_stderr)



================== manual estimator from samples ==================
manual estimates: {'ZZIIII': 1.04150390625, 'IIZZII': 0.8876953125, 'IIXXII': -0.01318359375}
library estimates: {'ZZIIII': 1.04150390625, 'IIZZII': 0.8876953125, 'IIXXII': -0.01318359375}
manual stderr: {'ZZIIII': 0.015904778115496124, 'IIZZII': 0.014824694405484234, 'IIXXII': 0.017090821268050237}
library stderr: {'ZZIIII': 0.015904778115496124, 'IIZZII': 0.014824694405484234, 'IIXXII': 0.017090821268050237}


# 7) 下一步
- 把 `circuit` 换成你自己的 `QuantumCircuit`。
- 调整 `shots_per_basis` 看方差变化。
- 在中层接口里替换 `prefer_chips` 和 `weights`，测试不同硬件策略。